# EEG Pipeline (metadata → features → quick checks)

Unisce:
- `create-df_features.ipynb` (pipeline completa: metadata + feature extraction)
- `dataframe.ipynb` (helper per dataframe / metadata)
- `feature_analysis.ipynb` (analisi e visualizzazione feature)

Data: 2026-03-05


## A) Helper / metadata dataframe (da `dataframe.ipynb`)


In [ ]:
# Robust imports and helper definitions for building EEG metadata DataFrame
from typing import Dict, Optional
from pathlib import Path
import json
import pandas as pd
import sys
import h5py
import re, unicodedata, difflib

# true_labels and canonicalization should already be defined in another cell;
# ensure they exist before calling build_eeg_dataframe when running the notebook.

def build_eeg_dataframe(h5_dir: Path, label_map_path: Optional[Path] = None) -> pd.DataFrame:
    """Build EEG metadata dataframe from H5 files.

    - Uses the last underscore in filename to split `subject_id` and `session_id`.
    - Loads an optional `label_map_path` (JSON) mapping labels to indices.
    - Skips malformed files and prints warnings.

    Args:
        h5_dir: Path to directory containing .h5 files (Path or str accepted)
        label_map_path: optional path to a JSON label->index map
    Returns:
        pd.DataFrame with rows for every epoch (subject_id, session_id, epoch_idx, label_name, label_idx, n_channels, n_samples, fs, path_h5)
    """
    h5_dir = Path(h5_dir)

    # Load label map if provided
    label2idx: Dict[str, int] = {}
    if label_map_path:
        try:
            with open(label_map_path, 'r', encoding='utf-8') as f:
                label2idx = json.load(f)
            print(f"Loaded label map with {len(label2idx)} entries")
        except FileNotFoundError:
            print(f"Label map not found at {label_map_path}; continuing without it")
        except Exception as e:
            print(f"Warning reading label map: {e}; continuing without it")

    rows = []
    skipped = []

    h5_files = sorted(h5_dir.glob('*.h5'))
    if not h5_files:
        print(f"⚠ No H5 files found in {h5_dir}")
        return pd.DataFrame()

    for file in h5_files:
        stem = file.stem
        if '_' not in stem:
            skipped.append(str(file))
            print(f"Skipping file with unexpected name (no underscore): {file.name}")
            continue

        # Use the last underscore as separator to support subject names that contain underscores
        subject_id, session_id = stem.rsplit('_', 1)

        try:
            with h5py.File(file, 'r') as f:
                if 'data' not in f or 'labels' not in f:
                    print(f"Warning: expected datasets missing in {file.name}")
                    skipped.append(str(file))
                    continue

                data = f['data']
                labels = f['labels'][:]  # type: ignore
                n_epochs, n_channels, n_samples = data.shape  # type: ignore

                for i, lbl_raw in enumerate(labels):  # type: ignore
                    # A canonicalize_label function is expected elsewhere in the notebook
                    try:
                        label_name = canonicalize_label(lbl_raw)
                    except Exception:
                        # Fallback: try simple decode
                        label_name = lbl_raw.decode('utf-8', 'ignore') if isinstance(lbl_raw, (bytes, bytearray)) else str(lbl_raw)

                    label_idx = label2idx.get(label_name, -1)

                    rows.append({
                        'subject_id': str(subject_id),
                        'session_id': str(session_id),
                        'epoch_idx': int(i),
                        'label_name': label_name,
                        'label_idx': int(label_idx) if isinstance(label_idx, (int, float)) else label_idx,
                        'n_channels': int(n_channels),
                        'n_samples': int(n_samples),
                        'fs': 256,
                        'path_h5': str(file)
                    })

            print(f"  ✓ Processed {file.name}: {n_epochs} epochs")
        except Exception as e:
            print(f"  ✗ Error processing {file.name}: {e}")
            skipped.append(str(file))
            continue

    df = pd.DataFrame(rows)
    if skipped:
        print(f"Skipped {len(skipped)} files; examples: {[Path(s).name for s in skipped[:10]]}")
    return df


## CONFIG

In [ ]:
# Path configuration function with user-specific bypass
from typing import Dict, List, Tuple
from pathlib import Path
import json
import pandas as pd
import sys

# Prefer an explicit project_root when running from a notebook
project_root = Path.cwd().parents[0]  # adjust if your notebook starts elsewhere
meta_csv = project_root / "data" / "interim" / "eeg_metadata.csv"
out = project_root / "data" / "interim" / "label2idx.json"

def get_data_paths(user_name: str = None) -> Dict[str, Path]:
    """
    Get data paths with user-specific overrides.
    
    The OneDrive path contains ALL subjects' H5 files:
    - Files are named as {subject_id}_{session_id}.h5
    - Example: 00_01.h5, 00_02.h5, ..., 01_01.h5, 01_02.h5, etc.
    - 5 acquisitions per subject (e.g., 00_01 to 00_05 for subject 00)
    
    For user 'daniele', uses OneDrive path (contains all subjects' data).
    For other users, uses default project paths.
    
    Args:
        user_name: Name of the user (e.g., 'daniele') - NOT the subject ID
    
    Returns:
        Dictionary with 'h5_dir' and 'eloc_path'
    """
    paths = {}
    
    # Electrode locations file - always in src/io
    paths['eloc_path'] = project_root / "src" / "io" / "ebneuro.elocs"
    
    # H5 data files path - user-specific bypass
    if user_name and user_name.lower() == 'daniele':
        # Bypass for user daniele: use OneDrive path (contains ALL subjects' data)
        # All H5 files (00_01.h5, 00_02.h5, 01_01.h5, etc.) are in this directory
        paths['h5_dir'] = Path('/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data')
        print(f"✓ Using daniele's OneDrive path (all subjects): {paths['h5_dir']}")
    else:
        # Default path for other users
        paths['h5_dir'] = project_root / "data" / "processed"
        print(f"✓ Using default path: {paths['h5_dir']}")
    
    # Verify paths exist
    if not paths['eloc_path'].exists():
        print(f"⚠ Warning: Electrode file not found at {paths['eloc_path']}")
    if not paths['h5_dir'].exists():
        print(f"⚠ Warning: H5 directory not found at {paths['h5_dir']}")
    
    return paths

# Configure paths - CHANGE THIS TO YOUR USER NAME
USER_NAME = 'daniele'  # Set to 'daniele' to use OneDrive path (all subjects), or None for default
paths = get_data_paths(USER_NAME)

print(f"\n📁 Configuration:")
print(f"   Electrode file: {paths['eloc_path']}")
print(f"   H5 data dir: {paths['h5_dir']}")
print(f"   Expected files: 00_01.h5, 00_02.h5, ..., 01_01.h5, etc.")

In [ ]:
import re, unicodedata, difflib
import h5py
import pandas as pd
from pathlib import Path
import json

## List of true labels in Italian
true_labels = [
    "arrivare","andare","aspettare","avere","capire","chiamare","chiedere","conoscere","dare","dire","dovere",
    "essere","fare","mettere","potere","prendere","sapere","sentire","trovare","venire","aprire","chiudere",
    "mangiare","bere","accendere","spegnere","volere","bene","si","no","più","poco","molto","sempre","adesso",
    "poi","male","sopra","sotto","destra","sinistra","avanti","indietro","oggi","domani","ieri","forse","prima",
    "perchè","anche","come","però","quindi","quando","dove","se","oppure","io","lui","lei","noi","voi","loro",
    "tu","questo","quello","buono","bello","cattivo","brutto","grande","piccolo","nuovo","vecchio","cosa","parte",
    "anno","casa","problema","aiuto","tempo","lavoro","persona","acqua","cibo","bisogno","donna","uomo","gruppo",
    "guerra","idea","macchina","mano","oggetto","telefono","computer","domanda","uno","mille","paura","ansia",
    "gioia","felicità","tristezza","serenità","amore","morte","bagno","dolore","riposo"
]

def _strip_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def _robust_decode(x) -> str:
    if isinstance(x, (bytes, bytearray)):
        for enc in ('utf-8', 'latin-1', 'cp1252'):
            try:
                s = x.decode(enc)
                break
            except Exception:
                continue
    else:
        s = str(x)

    s = s.replace('\x00', '').strip()

    if ('Ã' in s) or ('Â' in s):
        try:
            s = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
        except Exception:
            pass

    s = unicodedata.normalize('NFC', s)
    return s

def canonicalize_label(raw) -> str:
    s = _robust_decode(raw).lower()

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_img$', '', s)

    s = (s.replace('’', "'")
           .replace('‘', "'")
           .replace('“', '"')
           .replace('”', '"'))
    s = s.replace("''", "'").replace("´", "'").replace("`", "'").strip()

    fixes = {
        "perche": "perchè", "perche'": "perchè", "perch'e": "perchè", "perchà''": "perchè",
        "pero": "però", "piu": "più", "serenita": "serenità", "felicita": "felicità"
    }
    if s in fixes: 
        return fixes[s]

    if s in true_labels:
        return s

    s_ascii = _strip_diacritics(s)
    candidates_ascii = [_strip_diacritics(t) for t in true_labels]

    match1 = difflib.get_close_matches(s_ascii, candidates_ascii, n=1, cutoff=0.7)
    if match1:
        idx = candidates_ascii.index(match1[0])
        return true_labels[idx]

    match2 = difflib.get_close_matches(s, true_labels, n=1, cutoff=0.6)
    if match2:
        return match2[0]

    return s


In [ ]:

def build_eeg_dataframe(h5_dir, label_map_path):
    with open(label_map_path, "r", encoding="utf-8") as f:
        label2idx = json.load(f)

    rows = []
    for file in sorted(Path(h5_dir).glob("*.h5")):
        subject_id, session_id = file.stem.split("_")
        with h5py.File(file, "r") as f:
            data = f["data"]
            labels = f["labels"][:] # type: ignore
            n_epochs, n_channels, n_samples = data.shape # type: ignore

            for i, lbl_raw in enumerate(labels): # type: ignore
                label_name = canonicalize_label(lbl_raw)
                label_idx = label2idx.get(label_name, -1)
                rows.append({
                    "subject_id": subject_id,
                    "session_id": session_id,
                    "epoch_idx": i,
                    "label_name": label_name,
                    "label_idx": label_idx,
                    "n_channels": n_channels,
                    "n_samples": n_samples,
                    "fs": 256,
                    "path_h5": str(file)
                })

    df = pd.DataFrame(rows)
    return df


## 🧠 EEG Intelligent DataFrame – Overview

This DataFrame acts as the **central index** for your entire EEG dataset.  
Instead of duplicating raw data, it keeps **structured metadata** that describes every EEG epoch — including where it’s stored, which subject/session it belongs to, and its semantic label.

---

### 📁 Structure

Each row of the DataFrame corresponds to **one EEG epoch** and contains:

| Column | Description |
|:--------|:-------------|
| `subject_id` | ID of the participant (e.g. `11`) |
| `session_id` | Recording session (e.g. `S002`) |
| `epoch_idx` | Index of the epoch inside the `.h5` file |
| `label_name` | Semantic label (e.g. *"felicità"*, *"paura"*) |
| `label_idx` | Numerical class index from `label2idx.json` |
| `n_channels` | Number of EEG channels in that epoch |
| `n_samples` | Number of samples per channel |
| `path_h5` | Absolute path to the `.h5` file containing the signal |

---

### ⚙️ Purpose

The **EEG Intelligent DataFrame** serves as a lightweight, queryable "map" of your dataset.  
It allows you to:

1. **Explore** dataset composition and class balance.  
2. **Filter and retrieve** EEG signals on demand (by subject, session, or label).  
3. **Attach new features** (e.g., spectral power, connectivity metrics).  
4. **Build higher-level datasets** for PyTorch or PyTorch Geometric (graphs).  
5. **Visualize** or debug specific epochs without reloading everything.



## Build EEG Metadata DataFrame

In [ ]:

def build_eeg_dataframe(h5_dir: Path, label2idx: Dict[str, int] = None) -> pd.DataFrame:
    """
    Build EEG metadata dataframe from H5 files.
    
    Args:
        h5_dir: Directory containing H5 files
        label2idx: Dictionary mapping label names to indices (optional)
    
    Returns:
        DataFrame with metadata for each epoch
    """
    rows = []
    
    # Find all H5 files
    h5_files = sorted(h5_dir.glob("*.h5"))
    
    if not h5_files:
        print(f"⚠ No H5 files found in {h5_dir}")
        return pd.DataFrame()
    
    print(f"Found {len(h5_files)} H5 files")
    
    for file in h5_files:
        try:
            # Extract subject and session from filename
            parts = file.stem.split("_")
            subject_id = parts[0] if len(parts) > 0 else "unknown"
            session_id = parts[1] if len(parts) > 1 else "S001"
            
            with h5py.File(file, "r") as f:
                data = f["data"]
                labels = f["labels"][:] # type: ignore
                n_epochs, n_channels, n_samples = data.shape # type: ignore

                for i, lbl_raw in enumerate(labels): # type: ignore
                    label_name = canonicalize_label(lbl_raw)
                    label_idx = label2idx.get(label_name, -1) if label2idx else -1
                    
                    rows.append({
                        "subject_id": subject_id,
                        "session_id": session_id,
                        "epoch_idx": i,
                        "label_name": label_name,
                        "label_idx": label_idx,
                        "n_channels": n_channels,
                        "n_samples": n_samples,
                        "fs": 256,
                        "path_h5": str(file)
                    })
            
            print(f"  ✓ Processed {file.name}: {n_epochs} epochs")
        
        except Exception as e:
            print(f"  ✗ Error processing {file.name}: {e}")
            continue

    df = pd.DataFrame(rows)
    return df

# Build dataframe
print("\n📊 Building EEG metadata dataframe...")
# Try to load a precomputed label->index map; otherwise build a fallback from `true_labels`
interim_dir = project_root / "data" / "interim"
label2idx_path = interim_dir / "label2idx.json"
label2idx = None
if label2idx_path.exists():
    try:
        with open(label2idx_path, 'r', encoding='utf-8') as fh:
            label2idx = json.load(fh)
        # Canonicalize keys to match `canonicalize_label` outputs
        label2idx = {canonicalize_label(k): v for k, v in label2idx.items()}
        print(f"✓ Loaded label2idx from {label2idx_path} ({len(label2idx)} entries)")
    except Exception as e:
        print(f"⚠ Could not load label2idx.json: {e} - falling back to built mapping")
        label2idx = None
if label2idx is None:
    try:
        label2idx = {canonicalize_label(lbl): idx for idx, lbl in enumerate(true_labels)}
        print(f"✓ Built fallback label2idx from `true_labels` ({len(label2idx)} entries)")
    except Exception as e:
        print(f"⚠ Could not build fallback label2idx: {e}")
        label2idx = None

# Pass the mapping into the building function so label_idx is calculated correctly
meta_df_eeg = build_eeg_dataframe(paths['h5_dir'], label2idx=label2idx)

if not meta_df_eeg.empty:
    # Save to interim directory
    interim_dir.mkdir(parents=True, exist_ok=True)
    output_csv = interim_dir / "eeg_metadata.csv"
    meta_df_eeg.to_csv(output_csv, index=False)
    print(f"\n✓ Saved metadata to {output_csv}")
    print(f"\n📈 Dataset Summary:")
    print(f"   Total epochs: {len(meta_df_eeg)}")
    print(f"   Subjects: {meta_df_eeg['subject_id'].nunique()}")
    print(f"   Sessions: {meta_df_eeg['session_id'].nunique()}")
    print(f"   Labels: {meta_df_eeg['label_name'].nunique()}")
    print(f"   Channels: {meta_df_eeg['n_channels'].iloc[0]}")
    
    # Display first few rows
    display(meta_df_eeg.head(10))
else:
    print("⚠ No data found. Please check your H5 directory path.")

In [ ]:
# Number of occurrences of each label
print(meta_df_eeg["label_name"].value_counts().head(10))


In [ ]:
# Number of epochs per subject
print(meta_df_eeg.groupby("subject_id")["epoch_idx"].count())


In [ ]:
# Classes present in each subject
print(meta_df_eeg.groupby("subject_id")["label_name"].nunique())


In [ ]:
# Each epoch of subject 11 with class "felicità"
subset = meta_df_eeg.query("subject_id == '11' and label_name == 'felicità'")
print(subset)

## EEG Spectral Feature Extraction

For each EEG epoch (1.5 s, 256 Hz sampling rate), spectral features were extracted using **Welch’s Power Spectral Density (PSD)** method.  
The PSD represents how signal power is distributed across frequencies, measured in **µV²/Hz**, since the EEG amplitude was originally expressed in microvolts.

The PSD was integrated within canonical EEG frequency bands to obtain **band power values** in **µV²**:

| Band | Frequency Range (Hz) | Description |
|:-----|:---------------------|:-------------|
| **Delta (δ)** | 1 – 4 | Slow-wave activity, associated with deep sleep or low vigilance |
| **Theta (θ)** | 4 – 8 | Memory processes, drowsiness, limbic activation |
| **Alpha (α)** | 8 – 13 | Relaxed wakefulness, visual idling, eyes-closed resting state |
| **Beta (β)** | 13 – 30 | Motor activity, alertness, active cognitive processing |
| **Gamma (γ)** | 30 – 45 | Fast oscillations, sensory binding, high-level cognition |

### Computed Metrics

Each epoch is described by the following metrics:

| Metric | Definition | Unit | Description |
|:--------|:------------|:------|:-------------|
| **`total_power`** | \(\displaystyle P_\text{tot} = \int_{1}^{45} PSD(f)\,df\) | µV² | Total signal power across 1–45 Hz |
| **`delta`, `theta`, `alpha`, `beta`, `gamma`** | Band power from integration within each band | µV² | Absolute power per band |
| **`*_rel`** | \(\displaystyle P_\text{band} / P_\text{tot}\) | dimensionless | Relative contribution of each band to total power |
| **`alpha_beta_ratio`** | \(\displaystyle \frac{P_\alpha}{P_\beta}\) | dimensionless | Indicator of relaxation vs. activation |
| **`theta_alpha_ratio`** | \(\displaystyle \frac{P_\theta}{P_\alpha}\) | dimensionless | Cognitive fatigue or attentional engagement index |

The relative power values (`*_rel`) sum approximately to 1, representing the normalized spectral composition of each epoch.

### Physiological Interpretation

- **High delta/theta** → low vigilance or drowsy states  
- **High alpha** → relaxed or eyes-closed resting condition  
- **High beta** → active engagement or motor planning  
- **High gamma** → fast cognitive or perceptual integration  
- **Alpha/Beta ratio** → higher in calm or relaxed conditions  
- **Theta/Alpha ratio** → higher in fatigue or stress

---


## B) Pipeline completa (da `create-df_features.ipynb`)


# 🧠 Complete EEG Processing Pipeline

This notebook provides a complete end-to-end pipeline for processing EEG data:
1. Configuration of data paths (with user-specific bypasses)
2. Building EEG metadata dataframe
3. Extracting comprehensive features (temporal, spectral, functional)
4. Visualizing feature variations across epochs

## Path Configuration
- Electrode locations file: `src/io/ebneuro.elocs`
- H5 data files: 
  - For user 'daniele': OneDrive path contains all subjects (00_01.h5, 00_02.h5, 01_01.h5, etc.)
  - For other users: Default project path `data/processed/`

## Setup and Imports

In [ ]:
import re, unicodedata, difflib
import h5py
import pandas as pd
import numpy as np
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## Configuration: Path Setup with User-Specific Bypass

In [ ]:
# Path configuration function with user-specific bypass
def get_data_paths(user_name: str = None) -> Dict[str, Path]:
    """
    Get data paths with user-specific overrides.
    
    The OneDrive path contains ALL subjects' H5 files:
    - Files are named as {subject_id}_{session_id}.h5
    - Example: 00_01.h5, 00_02.h5, ..., 01_01.h5, 01_02.h5, etc.
    - 5 acquisitions per subject (e.g., 00_01 to 00_05 for subject 00)
    
    For user 'daniele', uses OneDrive path (contains all subjects' data).
    For other users, uses default project paths.
    
    Args:
        user_name: Name of the user (e.g., 'daniele') - NOT the subject ID
    
    Returns:
        Dictionary with 'h5_dir' and 'eloc_path'
    """
    paths = {}
    
    # Electrode locations file - always in src/io
    paths['eloc_path'] = project_root / "src" / "io" / "ebneuro.elocs"
    
    # H5 data files path - user-specific bypass
    if user_name and user_name.lower() == 'daniele':
        # Bypass for user daniele: use OneDrive path (contains ALL subjects' data)
        # All H5 files (00_01.h5, 00_02.h5, 01_01.h5, etc.) are in this directory
        paths['h5_dir'] = Path('/Users/danieleuras/Library/CloudStorage/OneDrive-PolitecnicodiMilano/File di Francesco Iacomi - h5/data')
        print(f"✓ Using daniele's OneDrive path (all subjects): {paths['h5_dir']}")
    else:
        # Default path for other users
        paths['h5_dir'] = project_root / "data" / "processed"
        print(f"✓ Using default path: {paths['h5_dir']}")
    
    # Verify paths exist
    if not paths['eloc_path'].exists():
        print(f"⚠ Warning: Electrode file not found at {paths['eloc_path']}")
    if not paths['h5_dir'].exists():
        print(f"⚠ Warning: H5 directory not found at {paths['h5_dir']}")
    
    return paths

# Configure paths - CHANGE THIS TO YOUR USER NAME
USER_NAME = 'daniele'  # Set to 'daniele' to use OneDrive path (all subjects), or None for default
paths = get_data_paths(USER_NAME)

print(f"\n📁 Configuration:")
print(f"   Electrode file: {paths['eloc_path']}")
print(f"   H5 data dir: {paths['h5_dir']}")
print(f"   Expected files: 00_01.h5, 00_02.h5, ..., 01_01.h5, etc.")

## Label Canonicalization Functions

In [ ]:
# List of true labels in Italian
true_labels = [
    "arrivare","andare","aspettare","avere","capire","chiamare","chiedere","conoscere","dare","dire","dovere",
    "essere","fare","mettere","potere","prendere","sapere","sentire","trovare","venire","aprire","chiudere",
    "mangiare","bere","accendere","spegnere","volere","bene","si","no","più","poco","molto","sempre","adesso",
    "poi","male","sopra","sotto","destra","sinistra","avanti","indietro","oggi","domani","ieri","forse","prima",
    "perchè","anche","come","però","quindi","quando","dove","se","oppure","io","lui","lei","noi","voi","loro",
    "tu","questo","quello","buono","bello","cattivo","brutto","grande","piccolo","nuovo","vecchio","cosa","parte",
    "anno","casa","problema","aiuto","tempo","lavoro","persona","acqua","cibo","bisogno","donna","uomo","gruppo",
    "guerra","idea","macchina","mano","oggetto","telefono","computer","domanda","uno","mille","paura","ansia",
    "gioia","felicità","tristezza","serenità","amore","morte","bagno","dolore","riposo"
]

def _strip_diacritics(s: str) -> str:
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def _robust_decode(x) -> str:
    if isinstance(x, (bytes, bytearray)):
        s = None
        for enc in ('utf-8', 'latin-1', 'cp1252'):
            try:
                s = x.decode(enc)
                break
            except Exception:
                continue
        # fallback if none of the decodings succeeded
        if s is None:
            s = x.decode('utf-8', 'ignore')
    else:
        s = str(x)

    s = s.replace('\x00', '').strip()

    if ('Ã' in s) or ('Â' in s):
        try:
            s = s.encode('latin-1', 'ignore').decode('utf-8', 'ignore')
        except Exception:
            pass

    s = unicodedata.normalize('NFC', s)
    return s

def canonicalize_label(raw) -> str:
    s = _robust_decode(raw).lower()

    s = re.sub(r'\s+', ' ', s).strip()
    s = re.sub(r'_img$', '', s)

    # Normalize various quote characters to simple ASCII quotes
    s = (s.replace("‘", "'")
           .replace("’", "'")
           .replace("‚", "'")
           .replace("“", '"')
           .replace("”", '"')
           .replace("„", '"'))

    s = s.replace("''", "'").replace("´", "'").replace("`", "'").strip()

    fixes = {
        "perche": "perchè", "perche'": "perchè", "perch'e": "perchè", "perchà''": "perchè",
        "pero": "però", "piu": "più", "serenita": "serenità", "felicita": "felicità"
    }
    if s in fixes:
        return fixes[s]

    if s in true_labels:
        return s

    s_ascii = _strip_diacritics(s)
    candidates_ascii = [_strip_diacritics(t) for t in true_labels]

    match1 = difflib.get_close_matches(s_ascii, candidates_ascii, n=1, cutoff=0.7)
    if match1:
        idx = candidates_ascii.index(match1[0])
        return true_labels[idx]

    match2 = difflib.get_close_matches(s, true_labels, n=1, cutoff=0.6)
    if match2:
        return match2[0]

    return s

print("✓ Label canonicalization functions loaded")

## Build EEG Metadata DataFrame

In [ ]:
def build_eeg_dataframe(h5_dir: Path, label2idx: Dict[str, int] = None) -> pd.DataFrame:
    """
    Build EEG metadata dataframe from H5 files.
    
    Args:
        h5_dir: Directory containing H5 files
        label2idx: Dictionary mapping label names to indices (optional)
    
    Returns:
        DataFrame with metadata for each epoch
    """
    rows = []
    
    # Find all H5 files
    h5_files = sorted(h5_dir.glob("*.h5"))
    
    if not h5_files:
        print(f"⚠ No H5 files found in {h5_dir}")
        return pd.DataFrame()
    
    print(f"Found {len(h5_files)} H5 files")
    
    for file in h5_files:
        try:
            # Extract subject and session from filename
            parts = file.stem.split("_")
            subject_id = parts[0] if len(parts) > 0 else "unknown"
            session_id = parts[1] if len(parts) > 1 else "S001"
            
            with h5py.File(file, "r") as f:
                data = f["data"]
                labels = f["labels"][:] # type: ignore
                n_epochs, n_channels, n_samples = data.shape # type: ignore

                for i, lbl_raw in enumerate(labels): # type: ignore
                    label_name = canonicalize_label(lbl_raw)
                    label_idx = label2idx.get(label_name, -1) if label2idx else -1
                    
                    rows.append({
                        "subject_id": subject_id,
                        "session_id": session_id,
                        "epoch_idx": i,
                        "label_name": label_name,
                        "label_idx": label_idx,
                        "n_channels": n_channels,
                        "n_samples": n_samples,
                        "fs": 256,
                        "path_h5": str(file)
                    })
            
            print(f"  ✓ Processed {file.name}: {n_epochs} epochs")
        
        except Exception as e:
            print(f"  ✗ Error processing {file.name}: {e}")
            continue

    df = pd.DataFrame(rows)
    return df

# Build dataframe
print("\n📊 Building EEG metadata dataframe...")
# Try to load a precomputed label->index map; otherwise build a fallback from `true_labels`
interim_dir = project_root / "data" / "interim"
label2idx_path = interim_dir / "label2idx.json"
label2idx = None
if label2idx_path.exists():
    try:
        with open(label2idx_path, 'r', encoding='utf-8') as fh:
            label2idx = json.load(fh)
        # Canonicalize keys to match `canonicalize_label` outputs
        label2idx = {canonicalize_label(k): v for k, v in label2idx.items()}
        print(f"✓ Loaded label2idx from {label2idx_path} ({len(label2idx)} entries)")
    except Exception as e:
        print(f"⚠ Could not load label2idx.json: {e} - falling back to built mapping")
        label2idx = None
if label2idx is None:
    try:
        label2idx = {canonicalize_label(lbl): idx for idx, lbl in enumerate(true_labels)}
        print(f"✓ Built fallback label2idx from `true_labels` ({len(label2idx)} entries)")
    except Exception as e:
        print(f"⚠ Could not build fallback label2idx: {e}")
        label2idx = None

# Pass the mapping into the building function so label_idx is calculated correctly
meta_df_eeg = build_eeg_dataframe(paths['h5_dir'], label2idx=label2idx)

if not meta_df_eeg.empty:
    # Save to interim directory
    interim_dir.mkdir(parents=True, exist_ok=True)
    output_csv = interim_dir / "eeg_metadata.csv"
    meta_df_eeg.to_csv(output_csv, index=False)
    print(f"\n✓ Saved metadata to {output_csv}")
    print(f"\n📈 Dataset Summary:")
    print(f"   Total epochs: {len(meta_df_eeg)}")
    print(f"   Subjects: {meta_df_eeg['subject_id'].nunique()}")
    print(f"   Sessions: {meta_df_eeg['session_id'].nunique()}")
    print(f"   Labels: {meta_df_eeg['label_name'].nunique()}")
    print(f"   Channels: {meta_df_eeg['n_channels'].iloc[0]}")
    
    # Display first few rows
    display(meta_df_eeg.head(10))
else:
    print("⚠ No data found. Please check your H5 directory path.")

## Extract Comprehensive Features

This section extracts temporal, spectral, and functional features from the EEG data.

In [ ]:
# Comprehensive extraction with resume/append support
# Set to True to run the comprehensive extractor from this notebook
RUN_COMPREHENSIVE = True  # <-- flip to True to execute
# If RESUME=True and output exists, we'll append subjects not yet processed
RESUME = False
# Parallel execution
PARALLEL = True  # Run per-subject workers in parallel
N_JOBS = 5  # Use 5 workers as requested
# Output path
out_csv = interim_dir / "comprehensive_features.csv"
eloc_path = paths['eloc_path']

if RUN_COMPREHENSIVE:
    print("Starting comprehensive extraction (notebook-controlled)...")
    import subprocess, concurrent.futures, os, sys, shutil
    try:
        from tqdm import tqdm
    except Exception:
        tqdm = None

    # Ensure subprocesses can import project package
    env = os.environ.copy()
    env['PYTHONPATH'] = str(project_root)
    cwd = str(project_root)

    # Load metadata (built earlier in the notebook)
    meta = pd.read_csv(interim_dir / "eeg_metadata.csv")
    meta['subject_id'] = meta['subject_id'].astype(str).str.strip()

    subjects = sorted(meta['subject_id'].unique())

    # Determine already processed subjects (for resume) by looking for per-subject files
    processed_subjects = set()
    per_subject_dir = interim_dir
    if out_csv.exists() and RESUME:
        try:
            prev = pd.read_csv(out_csv)
            processed_subjects = set(prev['subject_id'].astype(str).unique())
            print(f"Resuming: found {len(processed_subjects)} processed subjects in {out_csv}")
        except Exception as e:
            print(f"⚠ Could not read existing output for resume: {e}")
            processed_subjects = set()

    # If not resuming and file exists, overwrite
    if out_csv.exists() and not RESUME:
        print(f"Overwriting existing {out_csv}")
        out_csv.unlink()

    # Build list of subjects to process
    to_process = [s for s in subjects if s not in processed_subjects]
    if not to_process:
        print("No subjects to process.")
    else:
        print(f"Will process {len(to_process)} subjects (parallel={PARALLEL})")

        script_path = str(project_root / 'scripts' / 'run_comprehensive_subject.py')
        meta_csv_path = str(interim_dir / 'eeg_metadata.csv')
        eloc_arg = str(eloc_path)
        workers = N_JOBS if (N_JOBS not in (None, 0)) else max(1, (os.cpu_count() or 2) - 1)

        if PARALLEL:
            print(f"Launching up to {workers} parallel workers...")
            futures = {}
            with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as exe:
                for subj in to_process:
                    cmd = [sys.executable, '-m', 'scripts.run_comprehensive_subject', '--subject', str(subj), '--meta', meta_csv_path, '--out_dir', str(per_subject_dir), '--eloc', eloc_arg, '--fs', '256']
                    # Pass env and cwd so subprocess can locate project package
                    futures[exe.submit(subprocess.run, cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=cwd)] = subj

                iterator = concurrent.futures.as_completed(list(futures.keys()))
                if tqdm:
                    iterator = tqdm(iterator, total=len(futures), desc='subjects')
                for fut in iterator:
                    subj = futures[fut]
                    try:
                        res = fut.result()
                        header = f'--- Subject {subj} (ret={res.returncode}) ---'
                        print(header)
                        print(res.stdout)
                    except Exception as e:
                        print(f'✗ Worker for subject {subj} raised: {e}')
        else:
            # Serial fallback
            for subj in to_process:
                cmd = [sys.executable, '-m', 'scripts.run_comprehensive_subject', '--subject', str(subj), '--meta', meta_csv_path, '--out_dir', str(per_subject_dir), '--eloc', eloc_arg, '--fs', '256']
                print(f"Running: {' '.join(cmd)}")
                res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=cwd)
                print(res.stdout)

        # After workers finish, concatenate per-subject CSVs into final out_csv
        files = sorted(per_subject_dir.glob('comprehensive_subject_*.csv'))
        if not files:
            print("⚠ No per-subject outputs found — nothing to concatenate.")
        else:
            print(f"Concatenating {len(files)} per-subject files into {out_csv}...")
            df_list = [pd.read_csv(p) for p in files]
            df_all = pd.concat(df_list, ignore_index=True)
            df_all.to_csv(out_csv, index=False)
            print(f"✓ Saved combined features to {out_csv} ({len(df_all)} rows)")

            # Keep per-subject files as requested. Now move the combined file to data/processed
            processed_dir = project_root / 'data' / 'processed'
            processed_dir.mkdir(parents=True, exist_ok=True)
            dest = processed_dir / 'comprehensive_features_subjects.csv'
            try:
                shutil.move(str(out_csv), str(dest))
                print(f"✓ Moved combined features to {dest}")
            except Exception as e:
                print(f"⚠ Could not move combined file to {dest}: {e}")

    print("Comprehensive extraction (notebook) complete.")
else:
    print("RUN_COMPREHENSIVE is False — set it to True to execute comprehensive extraction from this cell.")

## Feature Visualization: Random Feature Variation Across Epochs

This section visualizes how a randomly selected feature varies across epochs.

In [ ]:
import os
df_features_path = "/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/comprehensive_features_subjects.csv"
if os.path.exists(df_features_path):
    df_features = pd.read_csv(df_features_path)
    print(f"✓ Loaded comprehensive features from {df_features_path} ({len(df_features)} rows)")
    display(df_features.head())


In [ ]:

def plot_random_feature_variation(df: pd.DataFrame, n_features: int = 3):
    """
    Plot variation of random features across epochs.
    
    Args:
        df: DataFrame with features
        n_features: Number of random features to plot
    """
    if df.empty:
        print("⚠ No data to visualize")
        return
    
    # Get numeric feature columns (exclude metadata)
    exclude_cols = ['subject_id', 'session_id', 'epoch_idx', 'channel', 'label_name']
    feature_cols = [col for col in df.columns if col not in exclude_cols and df[col].dtype in ['float64', 'int64']]
    
    if not feature_cols:
        print("⚠ No numeric features found")
        return
    
    # Select random features
    n_features = min(n_features, len(feature_cols))
    random_features = np.random.choice(feature_cols, size=n_features, replace=False)
    
    print(f"\n📊 Plotting {n_features} random features: {list(random_features)}")
    
    # Get a single subject and channel for cleaner visualization
    subject = df['subject_id'].iloc[0]
    channel = df['channel'].iloc[0] if 'channel' in df.columns else None
    
    # Filter data
    if channel:
        subset = df[(df['subject_id'] == subject) & (df['channel'] == channel)].copy()
        title_suffix = f"Subject: {subject}, Channel: {channel}"
    else:
        subset = df[df['subject_id'] == subject].copy()
        title_suffix = f"Subject: {subject}"
    
    if subset.empty:
        print("⚠ No data for visualization after filtering")
        return
    
    # Sort by epoch
    subset = subset.sort_values('epoch_idx')
    
    # Create subplots
    fig, axes = plt.subplots(n_features, 1, figsize=(14, 4*n_features))
    if n_features == 1:
        axes = [axes]
    
    colors = plt.cm.viridis(np.linspace(0, 1, n_features))
    
    for idx, (feature, ax, color) in enumerate(zip(random_features, axes, colors)):
        # Plot feature variation
        ax.plot(subset['epoch_idx'], subset[feature], 
                marker='o', linestyle='-', linewidth=2, 
                markersize=4, color=color, alpha=0.7)
        
        # Add mean line
        mean_val = subset[feature].mean()
        ax.axhline(y=mean_val, color='red', linestyle='--', 
                   linewidth=1.5, alpha=0.6, label=f'Mean: {mean_val:.4f}')
        
        # Styling
        ax.set_xlabel('Epoch Index', fontsize=11, fontweight='bold')
        ax.set_ylabel(feature, fontsize=11, fontweight='bold')
        ax.set_title(f'Feature: {feature}\n{title_suffix}', 
                     fontsize=12, fontweight='bold', pad=10)
        ax.grid(True, alpha=0.3)
        ax.legend(loc='upper right', fontsize=10)
        
        # Add statistics
        stats_text = f'Min: {subset[feature].min():.4f}\nMax: {subset[feature].max():.4f}\nStd: {subset[feature].std():.4f}'
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                fontsize=9, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle('Random Feature Variation Across Epochs', 
                 fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    
    # Save figure
    figures_dir = project_root / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    output_path = figures_dir / "random_feature_variation.png"
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved visualization to {output_path}")
    
    plt.show()

# Create visualization
if 'df_features' in locals() and not df_features.empty:
    plot_random_feature_variation(df_features, n_features=3)
elif not meta_df_eeg.empty:
    print("⚠ Using metadata for visualization (no features extracted yet)")
else:
    print("⚠ No data available for visualization")

## Channel Mapping

Build a mapping between H5 channel names and electrode location names.

In [ ]:
def build_channel_mapping(h5_path: Path, eloc_path: Path) -> pd.DataFrame:
    """
    Build channel mapping between H5 file and electrode locations.
    
    Args:
        h5_path: Path to an H5 file
        eloc_path: Path to electrode locations file
    
    Returns:
        DataFrame with channel mapping
    """
    try:
        # Get channel count from H5
        with h5py.File(h5_path, "r") as f:
            n_channels = f["data"].shape[1] # type: ignore
        h5_channels = [f"EEG{i+1}" for i in range(n_channels)]

        # Load electrode names from eloc file
        df_eloc = pd.read_csv(eloc_path, sep=r"\s+", header=None, engine="python", comment='#')
        eloc_channels = df_eloc.iloc[:, -1].astype(str).tolist()

        # Create mapping
        n_min = min(len(h5_channels), len(eloc_channels))
        mapping = pd.DataFrame({
            "channel_index": range(1, n_min + 1),
            "h5_name": h5_channels[:n_min],
            "eloc_name": eloc_channels[:n_min]
        })
        return mapping
    except Exception as e:
        print(f"⚠ Error building channel mapping: {e}")
        return pd.DataFrame()

# Build channel mapping if we have data
if not meta_df_eeg.empty and paths['eloc_path'].exists():
    first_h5 = Path(meta_df_eeg['path_h5'].iloc[0])
    if first_h5.exists():
        print("\n🗺️ Building channel mapping...")
        mapping_df = build_channel_mapping(first_h5, paths['eloc_path'])
        
        if not mapping_df.empty:
            # Save mapping
            mapping_csv = interim_dir / "channel_mapping.csv"
            mapping_df.to_csv(mapping_csv, index=False)
            print(f"✓ Saved channel mapping to {mapping_csv}")
            print(f"✓ Mapped {len(mapping_df)} channels")
            
            # Display mapping
            display(mapping_df.head(10))
    else:
        print(f"⚠ H5 file not found: {first_h5}")
else:
    print("⚠ Cannot build channel mapping: missing data or electrode file")

## Summary and Next Steps

### What This Notebook Does:
1. ✅ Configures paths with user-specific bypasses (e.g., user 'daniele' uses OneDrive path for all subjects)
2. ✅ Builds EEG metadata dataframe from H5 files (00_01.h5, 00_02.h5, 01_01.h5, etc.)
3. ✅ Extracts band power features (delta, theta, alpha, beta, gamma)
4. ✅ Visualizes random feature variations across epochs
5. ✅ Creates channel mapping between H5 and electrode locations

### Data Structure:
- **H5 files naming**: `{subject_id}_{session_id}.h5` (e.g., 00_01, 00_02, ..., 01_01, 01_02, ...)
- **5 acquisitions per subject**: Subject 00 has 00_01 to 00_05, subject 01 has 01_01 to 01_05, etc.
- **All subjects in one directory**: OneDrive path contains all subjects' data

### Output Files:
- `data/interim/eeg_metadata.csv` - Metadata for all epochs
- `data/interim/bandpowers.csv` - Extracted band power features
- `data/interim/channel_mapping.csv` - Channel name mapping
- `figures/random_feature_variation.png` - Feature variation plot

### Next Steps:
1. Extract additional features (temporal, functional)
2. Perform advanced visualizations
3. Build graph representations
4. Train machine learning models

## Label Statistics and Analysis

In [ ]:
if not meta_df_eeg.empty:
    print("\n📊 Label Distribution Analysis:\n")
    
    # Count occurrences
    label_counts = meta_df_eeg['label_name'].value_counts()
    print(f"Top 10 most frequent labels:")
    print(label_counts.head(10))
    
    # Epochs per subject
    print(f"\n📈 Epochs per subject:")
    print(meta_df_eeg.groupby('subject_id')['epoch_idx'].count())
    
    # Classes per subject
    print(f"\n🏷️ Unique labels per subject:")
    print(meta_df_eeg.groupby('subject_id')['label_name'].nunique())
    
    # Visualize label distribution
    plt.figure(figsize=(14, 6))
    label_counts.head(20).plot(kind='bar', color='steelblue', alpha=0.7)
    plt.title('Top 20 Label Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Label', fontsize=12)
    plt.ylabel('Count', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    # Save figure
    figures_dir = project_root / "figures"
    label_dist_path = figures_dir / "label_distribution.png"
    plt.savefig(label_dist_path, dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved label distribution plot to {label_dist_path}")
    
    plt.show()

## C) Feature analysis / plots (da `feature_analysis.ipynb`)


# EEG Feature Extraction and Visualization - Tutorial

This notebook demonstrates how to use the comprehensive feature extraction and visualization tools.

## Features Extracted:
- **Temporal**: Statistical measures, Hjorth parameters, zero-crossing rate, RMS
- **Spectral**: Band powers (delta, theta, alpha, beta, gamma), spectral entropy, dominant frequency
- **Functional**: Correlation-based connectivity, Phase Locking Value (PLV)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## Step 2: Explore the Features DataFrame

In [ ]:
# Display first few rows
import os
df_features_path = "/Users/danieleuras/Documents/GitHub/miralis-hypergraph-imagined-speech/data/processed/comprehensive_features_subjects.csv"
if os.path.exists(df_features_path):
    df_features = pd.read_csv(df_features_path)
    print(f"✓ Loaded comprehensive features from {df_features_path} ({len(df_features)} rows)")

print(df_features.head())

In [ ]:
# Display feature categories
temporal_features = [col for col in df_features.columns if col.startswith('temp_')]
spectral_features = [col for col in df_features.columns if col.startswith('spec_')]
functional_features = [col for col in df_features.columns if col.startswith('func_')]

print(f"Temporal features ({len(temporal_features)}): {temporal_features}")
print(f"\nSpectral features ({len(spectral_features)}): {spectral_features}")
print(f"\nFunctional features ({len(functional_features)}): {functional_features}")

In [ ]:
# Summary statistics
print("Dataset Summary:")
print(f"Number of subjects: {df_features['subject_id'].nunique()}")
print(f"Number of sessions: {df_features['session_id'].nunique()}")
print(f"Number of epochs: {df_features['epoch_idx'].nunique()}")
print(f"Number of channels: {df_features['channel'].nunique()}")
print(f"Number of labels: {df_features['label_name'].nunique()}")
print(f"\nTotal feature vectors: {len(df_features)}")

In [ ]:
# Elenca i soggetti che hanno registrazioni alla sessione 7
n=5
mask = df_features['session_id'] == n
subjects_sess7 = df_features.loc[mask, 'subject_id'].unique()

print(f"Soggetti con session_id == 7 ({len(subjects_sess7)}):")
print(list(subjects_sess7))

# Riepilogo per soggetto: numero di epoch, numero di canali e righe presenti per la sessione 7
summary_sess7 = (
    df_features[mask]
    .groupby('subject_id')
    .agg(
        epochs=('epoch_idx', 'nunique'),
        channels=('channel', 'nunique'),
        rows=('subject_id', 'size')
    )
    .sort_values(['epochs', 'channels', 'rows'], ascending=False)
)

print(f"\nRiepilogo per soggetto (epochs, channels, rows) per session_id == {n}:")
print(summary_sess7)

## Step 3: Basic Feature Analysis

In [ ]:
# Analyze power distribution across channels
channel_avg_power = df_features.groupby('channel')['spec_total_power'].mean().sort_values(ascending=False)

plt.figure(figsize=(14, 6))
plt.bar(range(len(channel_avg_power)), channel_avg_power.values, color='steelblue', alpha=0.7)
plt.xticks(range(len(channel_avg_power)), channel_avg_power.index, rotation=90, fontsize=8)
plt.xlabel('Channel', fontsize=12)
plt.ylabel('Average Total Power (µV²)', fontsize=12)
plt.title('Average Power Across Channels', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"\nTop 10 highest power channels:")
print(channel_avg_power.head(10))

In [ ]:
# Compare band powers
band_cols = ['spec_delta_rel', 'spec_theta_rel', 'spec_alpha_rel', 'spec_beta_rel', 'spec_gamma_rel']
band_averages = df_features[band_cols].mean()

plt.figure(figsize=(10, 6))
colors = ['#3498db', '#9b59b6', '#2ecc71', '#e74c3c', '#f39c12']
plt.bar(['Delta', 'Theta', 'Alpha', 'Beta', 'Gamma'], band_averages.values, color=colors, alpha=0.7)
plt.ylabel('Average Relative Power', fontsize=12)
plt.title('Average Relative Band Power Distribution', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Step 4: Generate Visualizations

In [ ]:
# Import visualization functions
from scripts.graphs.feature_visualizations import (
    plot_electrode_power_across_epochs,
    plot_top_electrodes_per_epoch,
    plot_feature_distributions,
    plot_feature_correlation_matrix
)

# Create output directory
output_dir = project_root / "figures" / "feature_visualizations"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Saving visualizations to: {output_dir}")

In [ ]:
# Visualization 1: Single electrode power variation
subject_id = df_features['subject_id'].iloc[0]
electrode = 'Cz'  # Central electrode

if electrode in df_features['channel'].values:
    plot_electrode_power_across_epochs(
        df_features, 
        electrode=electrode, 
        subject_id=subject_id,
        output_path=output_dir / f"power_variation_{electrode}.png"
    )
    print(f"✓ Created power variation plot for {electrode}")

In [ ]:
# Visualization 2: Top electrodes per epoch
plot_top_electrodes_per_epoch(
    df_features, 
    subject_id=subject_id, 
    top_n=5,
    output_path=output_dir / "top_electrodes_per_epoch.png"
)
print("✓ Created top electrodes heatmap")

In [ ]:
# Visualization 3: Feature distributions
plot_feature_distributions(
    df_features, 
    output_path=output_dir / "feature_distributions.png"
)
print("✓ Created feature distribution plots")

In [ ]:
# Visualization 4: Feature correlation matrix
plot_feature_correlation_matrix(
    df_features, 
    output_path=output_dir / "feature_correlation_matrix.png"
)
print("✓ Created feature correlation matrix")

## Step 5: Advanced Analysis - Label Comparison

In [ ]:
# Compare features across different labels
labels_to_compare = df_features['label_name'].value_counts().head(5).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

features_to_plot = [
    ('spec_alpha_rel', 'Alpha Relative Power'),
    ('spec_beta_rel', 'Beta Relative Power'),
    ('temp_hjorth_mobility', 'Hjorth Mobility'),
    ('func_mean_corr', 'Mean Correlation')
]

for i, (feat, title) in enumerate(features_to_plot):
    if feat in df_features.columns:
        data_to_plot = [df_features[df_features['label_name'] == label][feat].dropna().values 
                       for label in labels_to_compare]
        
        axes[i].boxplot(data_to_plot, labels=labels_to_compare)
        axes[i].set_ylabel(title, fontsize=10)
        axes[i].set_xticklabels(labels_to_compare, rotation=45, ha='right', fontsize=9)
        axes[i].set_title(f'{title} by Label', fontsize=11, fontweight='bold')
        axes[i].grid(True, alpha=0.3, axis='y')

plt.suptitle('Feature Comparison Across Top 5 Labels', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / "label_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print("✓ Created label comparison plot")

## Step 6: Export Summary Report

In [ ]:
# Create a summary report
summary = {
    'Dataset Info': {
        'Total Subjects': df_features['subject_id'].nunique(),
        'Total Sessions': df_features['session_id'].nunique(),
        'Total Epochs': df_features['epoch_idx'].nunique(),
        'Total Channels': df_features['channel'].nunique(),
        'Total Labels': df_features['label_name'].nunique(),
        'Total Feature Vectors': len(df_features)
    },
    'Feature Counts': {
        'Temporal Features': len(temporal_features),
        'Spectral Features': len(spectral_features),
        'Functional Features': len(functional_features),
        'Total Features': len(temporal_features) + len(spectral_features) + len(functional_features)
    },
    'Average Power Statistics': {
        'Mean Total Power': df_features['spec_total_power'].mean(),
        'Std Total Power': df_features['spec_total_power'].std(),
        'Max Total Power': df_features['spec_total_power'].max(),
        'Min Total Power': df_features['spec_total_power'].min()
    },
    'Band Power Averages': {
        'Delta': df_features['spec_delta_rel'].mean(),
        'Theta': df_features['spec_theta_rel'].mean(),
        'Alpha': df_features['spec_alpha_rel'].mean(),
        'Beta': df_features['spec_beta_rel'].mean(),
        'Gamma': df_features['spec_gamma_rel'].mean()
    }
}

# Print summary
import json
print(json.dumps(summary, indent=2))

# Save to file
with open(output_dir / 'feature_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Summary saved to {output_dir / 'feature_summary.json'}")

## Conclusion

This notebook demonstrated:
1. ✅ Feature extraction from EEG data (temporal, spectral, functional)
2. ✅ Creating a unified features dataframe
3. ✅ Visualizing power variations across electrodes and epochs
4. ✅ Identifying high-power electrodes
5. ✅ Comparing features across different labels
6. ✅ Analyzing feature distributions and correlations

All visualizations are saved in: `figures/feature_visualizations/`